# Assignment 4 – Optimizing Transformer Translation with Ray Tune & Optuna
**Baseline (100 epochs):** Time ≈ 1hr  | Final Loss ≈ 0.0974 | Baseline BLEU ≈ 71.47

**Goal:** Match/exceed BLEU ≥ 71.47 in ≤ 30 epochs using Ray Tune + OptunaSearch + ASHAScheduler.

## 1. Install Dependencies

In [1]:
!pip install ray[tune] optuna torch tqdm nltk -q

## 2. Imports

In [2]:
import os
import math
import time
import pickle
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

import ray
import ray.train
from ray import tune
from ray.tune import RunConfig
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import ASHAScheduler

nltk.download('punkt', quiet=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


## 3. Data Loading & Preprocessing

In [3]:
df = pd.read_csv('/kaggle/input/datasets/akankshakapil/english-hindi/English-Hindi - English-Hindi.tsv', sep='\t', header=None, names=["id1", "en", "id2", "hi"])
df = df[["en", "hi"]]
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Total pairs: {len(df)}")
df.head()

Total pairs: 13186


,en,hi
0,Muiriel is 20 now.,म्यूरियल अब बीस साल की हो गई है।
1,Muiriel is 20 now.,म्यूरियल अब बीस साल की है।
2,Education in this world disappoints me.,मैं इस दुनिया में शिक्षा पर बहुत निराश हूँ।
3,That won't happen.,वैसा नहीं होगा।
4,I miss you.,मुझें तुम्हारी याद आ रही है।


## 4. Vocabulary Builder

In [4]:
class Vocabulary:
    def __init__(self):
        self.stoi = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.itos = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.freq = {}

    def build(self, sentences, min_freq=2):
        for sent in sentences:
            for token in sent.split():
                self.freq[token] = self.freq.get(token, 0) + 1
        for token, count in self.freq.items():
            if count >= min_freq:
                idx = len(self.stoi)
                self.stoi[token] = idx
                self.itos[idx] = token

    def __len__(self):
        return len(self.stoi)

    def __getitem__(self, token):
        return self.stoi.get(token, self.stoi["<unk>"])


en_vocab = Vocabulary()
en_vocab.build(df["en"].tolist())

hi_vocab = Vocabulary()
hi_vocab.build(df["hi"].tolist())

print(f"English vocab size: {len(en_vocab)}")
print(f"Hindi vocab size:   {len(hi_vocab)}")

SRC_PAD_IDX = en_vocab["<pad>"]
TGT_PAD_IDX = hi_vocab["<pad>"]

English vocab size: 4354
Hindi vocab size:   4045


## 5. Encoding & Dataset

In [5]:
def encode_sentence(sentence, vocab, max_len=50):
    tokens = [vocab["<sos>"]] + [vocab[t] for t in sentence.split()] + [vocab["<eos>"]]
    tokens = tokens[:max_len]
    tokens += [vocab["<pad>"]] * (max_len - len(tokens))
    return tokens


class TranslationDataset(Dataset):
    def __init__(self, df, en_vocab, hi_vocab, max_len=50):
        self.en_sentences = df["en"].tolist()
        self.hi_sentences = df["hi"].tolist()
        self.en_vocab = en_vocab
        self.hi_vocab = hi_vocab
        self.max_len  = max_len

    def __len__(self):
        return len(self.en_sentences)

    def __getitem__(self, idx):
        src = encode_sentence(self.en_sentences[idx], self.en_vocab, self.max_len)
        tgt = encode_sentence(self.hi_sentences[idx], self.hi_vocab, self.max_len)
        return torch.tensor(src), torch.tensor(tgt)


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch  = torch.stack(src_batch)
    tgt_batch  = torch.stack(tgt_batch)
    # .contiguous() ensures memory is laid out correctly before slicing
    tgt_input  = tgt_batch[:, :-1].contiguous()
    tgt_output = tgt_batch[:, 1:].contiguous()
    return src_batch, tgt_input, tgt_output


MAX_LEN      = 50
full_dataset = TranslationDataset(df, en_vocab, hi_vocab, max_len=MAX_LEN)
print(f"Dataset size: {len(full_dataset)}")

Dataset size: 13186


## 6. Transformer Architecture

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512,
                 num_layers=6, num_heads=8, d_ff=2048, max_len=50, dropout=0.1):
        super().__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoding  = PositionalEncoding(d_model, max_len=max_len + 10, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(d_model, num_heads, d_ff, dropout, batch_first=True)
        decoder_layer = nn.TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, batch_first=True)
        self.encoder  = nn.TransformerEncoder(encoder_layer, num_layers)
        self.decoder  = nn.TransformerDecoder(decoder_layer, num_layers)
        self.fc_out   = nn.Linear(d_model, tgt_vocab_size)
        self.d_model  = d_model

    def make_src_key_padding_mask(self, src, pad_idx):
        return (src == pad_idx)

    def make_tgt_masks(self, tgt, pad_idx):
        T           = tgt.size(1)
        causal_mask = torch.triu(torch.ones(T, T, device=tgt.device), diagonal=1).bool()
        pad_mask    = (tgt == pad_idx)
        return causal_mask, pad_mask

    def forward(self, src, tgt, src_pad_idx, tgt_pad_idx):
        src_key_padding_mask           = self.make_src_key_padding_mask(src, src_pad_idx)
        tgt_mask, tgt_key_padding_mask = self.make_tgt_masks(tgt, tgt_pad_idx)

        src_emb = self.pos_encoding(self.src_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoding(self.tgt_embedding(tgt) * math.sqrt(self.d_model))

        memory = self.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)
        output = self.decoder(
            tgt_emb, memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )
        return self.fc_out(output)

## 7. Evaluation Helpers

In [7]:
def translate_sentence(model, sentence, en_vocab, hi_vocab, max_len=50, device=DEVICE):
    model.eval()
    tokens     = encode_sentence(sentence, en_vocab, max_len=max_len)
    src_tensor = torch.tensor(tokens).unsqueeze(0).to(device)
    tgt_tokens = [hi_vocab["<sos>"]]
    for _ in range(max_len):
        tgt_tensor = torch.tensor(tgt_tokens).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(src_tensor, tgt_tensor, SRC_PAD_IDX, TGT_PAD_IDX)
        next_token = output[0, -1].argmax().item()
        tgt_tokens.append(next_token)
        if next_token == hi_vocab["<eos>"]:
            break
    translated = [hi_vocab.itos[idx] for idx in tgt_tokens[1:-1]]
    return ' '.join(translated)


VAL_DATASET = [
    ("I love you.",                 "मैं तुमसे प्यार करता हूँ।"),
    ("How are you?",                "आप कैसे हैं?"),
    ("You should sleep.",           "आपको सोना चाहिए।"),
    ("Maybe Tom doesn't love you.", "टॉम शायद तुमसे प्यार नहीं करता है।"),
    ("Let me tell Tom.",            "मुझे टॉम को बताने दीजिए।")
]

smoothie = SmoothingFunction().method4


def evaluate_bleu(model, dataset=VAL_DATASET, max_len=50, device=DEVICE):
    references, hypotheses = [], []
    for en_sent, hi_sent in dataset:
        pred = translate_sentence(model, en_sent, en_vocab, hi_vocab, max_len, device)
        hypotheses.append(pred.split())
        references.append([hi_sent.split()])
    return corpus_bleu(references, hypotheses, smoothing_function=smoothie)

## 8. `train_tune` — Ray Tune Compatible Training Function

Key fixes vs original:
- Uses `.reshape()` instead of `.view()` to handle non-contiguous tensors
- Reports metrics via `ray.train.report()` each epoch
- Adds gradient clipping and cosine LR scheduler for faster convergence

In [8]:
def train_tune(config):
    """
    Ray Tune compatible training function.
    Hyperparameters received via config dict:
        lr, batch_size, num_heads, d_ff, dropout, num_layers
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── Hyperparameters from config ─────────────────────────────────────────
    lr         = config["lr"]
    batch_size = config["batch_size"]
    num_heads  = config["num_heads"]
    d_ff       = config["d_ff"]
    dropout    = config["dropout"]
    num_layers = config["num_layers"]
    num_epochs = config.get("num_epochs", 30)
    D_MODEL    = 512   # fixed — divisible by both 4 and 8

    # ── DataLoader ──────────────────────────────────────────────────────────
    dataset = TranslationDataset(df, en_vocab, hi_vocab, max_len=MAX_LEN)
    loader  = DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=0,
        pin_memory=(device.type == "cuda")
    )

    # ── Model ───────────────────────────────────────────────────────────────
    model = Transformer(
        src_vocab_size=len(en_vocab),
        tgt_vocab_size=len(hi_vocab),
        d_model=D_MODEL,
        num_layers=num_layers,
        num_heads=num_heads,
        d_ff=d_ff,
        max_len=MAX_LEN,
        dropout=dropout
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD_IDX)

    # ── Training loop ───────────────────────────────────────────────────────
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for src, tgt_input, tgt_output in loader:
            src        = src.to(device)
            tgt_input  = tgt_input.to(device)
            tgt_output = tgt_output.to(device)

            output = model(src, tgt_input, SRC_PAD_IDX, TGT_PAD_IDX)

            # reshape (not view) — safely handles non-contiguous tensors
            output       = output.reshape(-1, output.shape[-1])
            tgt_out_flat = tgt_output.reshape(-1)

            loss = criterion(output, tgt_out_flat)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(loader)

        # Report to Ray Tune every epoch — ASHA uses this to prune bad trials
        tune.report({"loss": avg_loss, "epoch": epoch + 1})

## 9. Search Space

Tuning **6 hyperparameters** (requirement is ≥ 4):

| Hyperparameter | Type | Range / Choices |
|---|---|---|
| `lr` | log-uniform | 1e-5 – 1e-3 |
| `batch_size` | choice | 32, 64, 128 |
| `num_heads` | choice | 4, 8 |
| `d_ff` | choice | 1024, 2048 |
| `dropout` | uniform | 0.1 – 0.4 |
| `num_layers` | choice | 4, 6 |

In [9]:
search_space = {
    "lr":         tune.loguniform(1e-5, 1e-3),
    "batch_size": tune.choice([32, 64, 128]),
    "num_heads":  tune.choice([4, 8]),          # D_MODEL=512 divisible by both
    "d_ff":       tune.choice([1024, 2048]),
    "dropout":    tune.uniform(0.1, 0.4),
    "num_layers": tune.choice([4, 6]),
    "num_epochs": 30,                           # fixed cap per trial
}

## 10. Configure Optuna + ASHA and Run the Sweep

In [10]:
if not ray.is_initialized():
    ray.init(
        ignore_reinit_error=True,
        num_cpus=4,
        num_gpus=1
    )

optuna_search = OptunaSearch(metric="loss", mode="min")

asha_scheduler = ASHAScheduler(
    metric="loss",
    mode="min",
    max_t=30,
    grace_period=5,
    reduction_factor=2
)

tuner = tune.Tuner(
    tune.with_resources(
        train_tune,
        resources={"cpu": 2, "gpu": 0.5}  # 2 trials share the 1 GPU
    ),
    tune_config=tune.TuneConfig(
        search_alg=optuna_search,
        scheduler=asha_scheduler,
        num_samples=20,
    ),
    param_space=search_space,
    run_config=RunConfig(
        name="en_to_hi_optuna_sweep",
    )
)

print("Starting Ray Tune sweep — 20 trials, max 30 epochs each...")
sweep_start = time.time()
results     = tuner.fit()
sweep_time  = time.time() - sweep_start
print(f"\nSweep finished in {sweep_time/60:.1f} minutes")

(train_tune pid=2655) [2026-03-18 16:53:52,244 E 2655 2677] core_worker_process.cc:842: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14 [repeated 4x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(train_tune pid=2694) [2026-03-18 16:53:58,326 E 2694 2723] core_worker_process.cc:842: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(train_tune pid=2830) [2026-03-18 17:01:54,234 E 2830 2854] core_worker_process.cc:842: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent


Sweep finished in 234.9 minutes


## 11. Analyse Results

In [11]:
best_result  = results.get_best_result(metric="loss", mode="min")
best_config  = best_result.config
best_metrics = best_result.metrics

print("=" * 55)
print("BEST CONFIGURATION FOUND BY OPTUNA")
print("=" * 55)
for k, v in best_config.items():
    print(f"  {k:15s}: {v}")
print(f"\nBest trial final loss : {best_metrics['loss']:.4f}")
print(f"Achieved at epoch     : {best_metrics.get('epoch', '?')}")

# All trials summary
df_results = results.get_dataframe()
cols = ["loss", "epoch",
        "config/lr", "config/batch_size", "config/num_heads",
        "config/d_ff", "config/dropout", "config/num_layers"]
print("\nAll trial results (sorted by loss):")
print(df_results[[c for c in cols if c in df_results.columns]]
      .sort_values("loss").to_string(index=False))

BEST CONFIGURATION FOUND BY OPTUNA
  lr             : 0.00018823892905166936
  batch_size     : 64
  num_heads      : 8
  d_ff           : 1024
  dropout        : 0.10470617591008506
  num_layers     : 6
  num_epochs     : 30

Best trial final loss : 0.1696
Achieved at epoch     : 30

All trial results (sorted by loss):
    loss  epoch  config/lr  config/batch_size  config/num_heads  config/d_ff  config/dropout  config/num_layers
0.169581     30   0.000188                 64                 8         1024        0.104706                  6
0.214955     30   0.000210                 64                 8         1024        0.138191                  6
0.233644     30   0.000169                 64                 8         1024        0.101618                  4
0.307035     30   0.000145                 64                 8         1024        0.101516                  6
0.490493     30   0.000137                 64                 8         1024        0.109711                  4
1.3682

## 12. Retrain Best Model & Evaluate BLEU

In [12]:
def train_best_model(config, num_epochs=30, save_path="rollno_ass_4_best_model.pth"):
    """
    Full retrain with best config. Returns (model, final_loss, bleu, time).
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    lr         = config["lr"]
    batch_size = config["batch_size"]
    num_heads  = config["num_heads"]
    d_ff       = config["d_ff"]
    dropout    = config["dropout"]
    num_layers = config["num_layers"]
    D_MODEL    = 512

    dataset = TranslationDataset(df, en_vocab, hi_vocab, max_len=MAX_LEN)
    loader  = DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=0
    )

    model = Transformer(
        src_vocab_size=len(en_vocab),
        tgt_vocab_size=len(hi_vocab),
        d_model=D_MODEL,
        num_layers=num_layers,
        num_heads=num_heads,
        d_ff=d_ff,
        max_len=MAX_LEN,
        dropout=dropout
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD_IDX)

    start_time   = time.time()
    epoch_losses = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        loop = tqdm(loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=False)

        for src, tgt_input, tgt_output in loop:
            src        = src.to(device)
            tgt_input  = tgt_input.to(device)
            tgt_output = tgt_output.to(device)

            output = model(src, tgt_input, SRC_PAD_IDX, TGT_PAD_IDX)

            # reshape instead of view — handles non-contiguous tensors safely
            output       = output.reshape(-1, output.shape[-1])
            tgt_out_flat = tgt_output.reshape(-1)

            loss = criterion(output, tgt_out_flat)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        scheduler.step()
        avg = epoch_loss / len(loader)
        epoch_losses.append(avg)
        print(f"Epoch {epoch+1:3d}/{num_epochs} | Loss: {avg:.4f}")

    total_time = time.time() - start_time
    final_loss = epoch_losses[-1]

    bleu = evaluate_bleu(model, VAL_DATASET, MAX_LEN, device)
    print(f"\n🌐 BLEU Score : {bleu * 100:.2f}")
    print(f"⏱  Total time : {total_time/60:.1f} min")
    print(f"📉 Final loss : {final_loss:.4f}")

    # Save model weights
    torch.save(model.state_dict(), save_path)
    print(f"\n✅ Best model saved to: {save_path}")

    # Save vocabs
    with open("en_vocab.pkl", "wb") as f:
        pickle.dump(en_vocab, f)
    with open("hi_vocab.pkl", "wb") as f:
        pickle.dump(hi_vocab, f)

    return model, final_loss, bleu, total_time


print("Retraining best model with config:")
for k, v in best_config.items():
    print(f"  {k}: {v}")

best_model, final_loss, bleu_score, total_time = train_best_model(
    best_config, num_epochs=30, save_path="rollno_ass_4_best_model.pth"
)

Retraining best model with config:
  lr: 0.00018823892905166936
  batch_size: 64
  num_heads: 8
  d_ff: 1024
  dropout: 0.10470617591008506
  num_layers: 6
  num_epochs: 30


Epoch   1/30 | Loss: 5.3252


Epoch   2/30 | Loss: 4.3389


Epoch   3/30 | Loss: 3.7460


Epoch   4/30 | Loss: 3.2583


...

Epoch  27/30 | Loss: 0.1795


Epoch  28/30 | Loss: 0.1752


Epoch  29/30 | Loss: 0.1691


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch  30/30 | Loss: 0.1668

🌐 BLEU Score : 79.03
⏱  Total time : 26.7 min
📉 Final loss : 0.1668

✅ Best model saved to: rollno_ass_4_best_model.pth


## 13. Final Results Summary

In [16]:
print("=" * 55)
print("   BASELINE vs BEST MODEL COMPARISON")
print("=" * 55)
print(f"{'Metric':<25} {'Baseline':>12} {'Best Model':>12}")
print("-" * 55)
print(f"{'Epochs':<25} {'100':>12} {'30':>12}")
print(f"{'Total Time (min)':<25} {'~ 60':>12} {total_time/60:>12.1f}")
print(f"{'Final Loss':<25} {'0.0978':>12} {final_loss:>12.4f}")
print(f"{'BLEU Score':<25} {'71.47':>12} {bleu_score*100:>12.2f}")
print("=" * 55)

print("\n📌 Best Hyperparameter Configuration:")
for k, v in best_config.items():
    if k != "num_epochs":
        print(f"   {k:15s}: {v}")

   BASELINE vs BEST MODEL COMPARISON
Metric                        Baseline   Best Model
-------------------------------------------------------
Epochs                             100           30
Total Time (min)                  ~ 60         26.7
Final Loss                      0.0978       0.1668
BLEU Score                       71.47        79.03

📌 Best Hyperparameter Configuration:
   lr             : 0.00018823892905166936
   batch_size     : 64
   num_heads      : 8
   d_ff           : 1024
   dropout        : 0.10470617591008506
   num_layers     : 6


## 14. Sample Translations with Best Model

In [14]:
example_sentences = [
    "I love you.",
    "What is your name?",
    "How are you?",
    "The weather is nice today.",
    "She is a good teacher."
]

print("Sample Translations (Best Model):\n")
for sentence in example_sentences:
    translation = translate_sentence(best_model, sentence, en_vocab, hi_vocab)
    print(f"🗣  English : {sentence}")
    print(f"🇮🇳 Hindi   : {translation}\n")

Sample Translations (Best Model):

🗣  English : I love you.
🇮🇳 Hindi   : मैं तुमसे प्यार करता हूँ।

🗣  English : What is your name?
🇮🇳 Hindi   : तुम्हारा नाम क्या है?

🗣  English : How are you?
🇮🇳 Hindi   : आप कैसे हो?

🗣  English : The weather is nice today.
🇮🇳 Hindi   : आज मौसम बहुत अच्छा है।

🗣  English : She is a good teacher.
🇮🇳 Hindi   : वह एक अच्छी अध्यापिका है.

